# 05 — Property Value Processed Metrics

**Phase:** Processed Output Preparation  
**Notebook goal:** Prepare small, validated processed outputs from the City of Vancouver Property Tax Report that summarise assessed property value changes, for later comparison with residential permit activity.

---

## Purpose

This notebook computes four derived variables that measure year-over-year change in assessed property values:

- `current_total_assessed_value` — land + improvement assessed value for the current tax year
- `previous_total_assessed_value` — land + improvement assessed value for the prior tax year
- `absolute_value_change` — difference between current and previous totals
- `percentage_value_change` — that difference expressed as a percentage of the prior-year total

These variables were developed and validated individually in **Notebook 02**. This notebook consolidates that logic into a reusable function, validates it on a 1,000-row sample, and defines the planned processed output files.

> **Important caveat:** BC Assessment assessed values are not MLS sale prices, transaction prices, or market values. They are administrative valuations used as the basis for property tax calculations in British Columbia. Year-over-year changes in assessed value do not directly measure market appreciation or depreciation.

> **Scope:** This notebook does not yet process the full raw dataset. Sample logic is verified here first. Full-dataset processing will be added in a later step once the outputs and exports are confirmed correct on the sample.

## 1. Imports and Paths

We use `pathlib` for cross-platform path handling, `pandas` for all tabular operations, and `math` for the infinity check. All paths are derived from `PROJECT_ROOT` so the notebook works regardless of where it is launched from.

In [1]:
from pathlib import Path
import pandas as pd
import math

# Project root is one level above the notebooks/ directory
PROJECT_ROOT       = Path(__file__).resolve().parent.parent if '__file__' in dir() else Path.cwd().parent
RAW_DATA_PATH      = PROJECT_ROOT / 'data' / 'raw' / 'property_tax_report_raw.csv'
PROCESSED_DATA_DIR = PROJECT_ROOT / 'data' / 'processed'

print(f'PROJECT_ROOT       : {PROJECT_ROOT}')
print(f'RAW_DATA_PATH      : {RAW_DATA_PATH}')
print(f'PROCESSED_DATA_DIR : {PROCESSED_DATA_DIR}')
print(f'Raw file exists    : {RAW_DATA_PATH.exists()}')
print(f'Processed dir exists: {PROCESSED_DATA_DIR.exists()}')

PROJECT_ROOT       : c:\Users\User\Documents\vancouver-property-value-housing-supply-analysis
RAW_DATA_PATH      : c:\Users\User\Documents\vancouver-property-value-housing-supply-analysis\data\raw\property_tax_report_raw.csv
PROCESSED_DATA_DIR : c:\Users\User\Documents\vancouver-property-value-housing-supply-analysis\data\processed
Raw file exists    : True
Processed dir exists: True


## 2. Constants

Source column names are confirmed in Notebook 02. Storing them as constants means a source rename requires changing only one line here, not every occurrence throughout the notebook.

In [2]:
# Source column names — confirmed present in Notebook 02
COL_CURR_LAND        = 'CURRENT_LAND_VALUE'
COL_CURR_IMPROVEMENT = 'CURRENT_IMPROVEMENT_VALUE'
COL_PREV_LAND        = 'PREVIOUS_LAND_VALUE'
COL_PREV_IMPROVEMENT = 'PREVIOUS_IMPROVEMENT_VALUE'
COL_PID              = 'PID'

# Derived variable names
COL_CURR_TOTAL  = 'current_total_assessed_value'
COL_PREV_TOTAL  = 'previous_total_assessed_value'
COL_ABS_CHANGE  = 'absolute_value_change'
COL_PCT_CHANGE  = 'percentage_value_change'

# Sample size — full dataset not processed in this notebook
SAMPLE_ROWS = 1_000

print('Constants defined.')

Constants defined.


## 3. Sample Load

The raw CSV file is approximately 443 MB, which makes loading the full dataset slow during development. We load only 1,000 rows here to verify column names and logic.

Key `read_csv` arguments:
- **`sep=';'`** — the file uses semicolons as delimiters, not commas.
- **`nrows=SAMPLE_ROWS`** — stops reading after 1,000 rows.
- **`encoding='utf-8-sig'`** — strips the byte-order mark (BOM) from the first column name.
- **`low_memory=False`** — reads the full sample before assigning `dtype`s, reducing mixed-type warnings.

In [3]:
df = pd.read_csv(
    RAW_DATA_PATH,
    sep=';',
    nrows=SAMPLE_ROWS,
    encoding='utf-8-sig',
    low_memory=False,
)

print(f'Shape   : {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Columns : {list(df.columns)}')
print()
display(df.head(5))

Shape   : 1,000 rows × 30 columns
Columns : ['PID', 'LEGAL_TYPE', 'FOLIO', 'LAND_COORDINATE', 'ZONING_DISTRICT', 'ZONING_CLASSIFICATION', 'LOT', 'PLAN', 'BLOCK', 'DISTRICT_LOT', 'FROM_CIVIC_NUMBER', 'TO_CIVIC_NUMBER', 'STREET_NAME', 'PROPERTY_POSTAL_CODE', 'NARRATIVE_LEGAL_LINE1', 'NARRATIVE_LEGAL_LINE2', 'NARRATIVE_LEGAL_LINE3', 'NARRATIVE_LEGAL_LINE4', 'NARRATIVE_LEGAL_LINE5', 'CURRENT_LAND_VALUE', 'CURRENT_IMPROVEMENT_VALUE', 'TAX_ASSESSMENT_YEAR', 'PREVIOUS_LAND_VALUE', 'PREVIOUS_IMPROVEMENT_VALUE', 'YEAR_BUILT', 'BIG_IMPROVEMENT_YEAR', 'TAX_LEVY', 'NEIGHBOURHOOD_CODE', 'REPORT_YEAR', 'NOTE']



,PID,LEGAL_TYPE,FOLIO,LAND_COORDINATE,ZONING_DISTRICT,ZONING_CLASSIFICATION,LOT,PLAN,BLOCK,DISTRICT_LOT,...,CURRENT_IMPROVEMENT_VALUE,TAX_ASSESSMENT_YEAR,PREVIOUS_LAND_VALUE,PREVIOUS_IMPROVEMENT_VALUE,YEAR_BUILT,BIG_IMPROVEMENT_YEAR,TAX_LEVY,NEIGHBOURHOOD_CODE,REPORT_YEAR,NOTE
0,013-172-549,LAND,830212830000,83021283,R1-1,Residential Inclusive,4,VAP3089,A,327,...,767000.0,2023.0,1161192.0,799000.0,2017.0,2017.0,6678.66,17,2023,NaN
1,030-022-843,STRATA,320718950602,32071895,CD-1 (545),Comprehensive Development,602,EPS3434,6,36,...,322000.0,2023.0,507000.0,306000.0,2016.0,2016.0,2469.26,23,2023,NaN
2,027-537-978,STRATA,592118060076,59211806,DD,Comprehensive Development,76,BCS2936,NaN,185,...,427000.0,2023.0,800000.0,415000.0,2008.0,2008.0,3392.44,26,2023,NaN
3,028-763-017,STRATA,670026450025,67002645,C-2,Commercial,25,BCS4338,148,540,...,347000.0,2023.0,1009000.0,311000.0,2012.0,2012.0,4183.51,1,2023,NaN
4,028-763-050,STRATA,670026450029,67002645,C-2,Commercial,29,BCS4338,148,540,...,182000.0,2023.0,427000.0,175000.0,2012.0,2012.0,1876.98,1,2023,NaN


## 4. Column Verification

Before any feature engineering, we confirm that each required source column is present. These names were established in Notebook 02; we reuse them exactly.

In [4]:
required_columns = [COL_PID,COL_CURR_LAND,COL_CURR_IMPROVEMENT,COL_PREV_LAND,COL_PREV_IMPROVEMENT,]

all_present = True
for col in required_columns:
    status = 'FOUND  ' if col in df.columns else 'MISSING'
    print(f'  {status}: {col}')
    if 'MISSING' in status:
        all_present = False

print()
if all_present:
    print('All required columns present. Proceeding.')
else:
    print('WARNING: One or more required columns are missing. Do not proceed until resolved.')

  FOUND  : PID
  FOUND  : CURRENT_LAND_VALUE
  FOUND  : CURRENT_IMPROVEMENT_VALUE
  FOUND  : PREVIOUS_LAND_VALUE
  FOUND  : PREVIOUS_IMPROVEMENT_VALUE

All required columns present. Proceeding.


## 5. Feature Engineering Function

The four derived variables are bundled into a single reusable function. This makes it straightforward to apply the same logic to the full dataset later without repeating code.

### Missing-value and safety rules

- **Totals** use `.sum(axis=1, min_count=2)`: both source components must be non-null for the total to be non-null. A partial total (one component present, one missing) is not computed — it would silently understate the full assessed value.
- **Absolute change** uses standard subtraction (`-`). Pandas propagates `NaN` through subtraction automatically, so any row missing either total produces a null change.
- **Percentage change** is computed only for rows where `previous_total_assessed_value` is strictly positive. Rows with a null, zero, or negative previous total receive a null percentage. This avoids `inf` from division by zero and sign-inverted results from a negative denominator.

In [5]:
def compute_value_change_features(df: pd.DataFrame) -> pd.DataFrame:
    """Return a copy of df with four assessed-value change columns appended."""
    out = df.copy()

    # Current and previous assessed totals
    out[COL_CURR_TOTAL] = out[[COL_CURR_LAND, COL_CURR_IMPROVEMENT]].sum(axis=1, min_count=2)
    out[COL_PREV_TOTAL] = out[[COL_PREV_LAND, COL_PREV_IMPROVEMENT]].sum(axis=1, min_count=2)

    # Absolute year-over-year change; NaN propagates automatically
    out[COL_ABS_CHANGE] = out[COL_CURR_TOTAL] - out[COL_PREV_TOTAL]

    # Percentage change — only for rows with a strictly positive previous total
    out[COL_PCT_CHANGE] = float('nan')
    eligible = out[COL_PREV_TOTAL].notna() & (out[COL_PREV_TOTAL] > 0) & out[COL_ABS_CHANGE].notna()
    out.loc[eligible, COL_PCT_CHANGE] = (
        out.loc[eligible, COL_ABS_CHANGE] / out.loc[eligible, COL_PREV_TOTAL] * 100
    )

    return out


print('Function defined: compute_value_change_features')

Function defined: compute_value_change_features


### Apply the function to the sample

In [6]:
df = compute_value_change_features(df)

derived_cols = [COL_CURR_TOTAL, COL_PREV_TOTAL, COL_ABS_CHANGE, COL_PCT_CHANGE]
for col in derived_cols:
    print(f'  {col}: dtype={df[col].dtype}')

print()
display(df[[COL_PID] + derived_cols].head(10))

  current_total_assessed_value: dtype=float64
  previous_total_assessed_value: dtype=float64
  absolute_value_change: dtype=float64
  percentage_value_change: dtype=float64



,PID,current_total_assessed_value,previous_total_assessed_value,absolute_value_change,percentage_value_change
0,013-172-549,2017000.0,1960192.0,56808.0,2.898083
1,030-022-843,888000.0,813000.0,75000.0,9.225092
2,027-537-978,1220000.0,1215000.0,5000.0,0.411523
3,028-763-017,1504477.0,1320000.0,184477.0,13.975530
4,028-763-050,675000.0,602000.0,73000.0,12.126246
5,027-591-328,6149000.0,5962000.0,187000.0,3.136531
6,027-549-356,810000.0,768000.0,42000.0,5.468750
7,030-019-401,947000.0,860000.0,87000.0,10.116279
8,030-019-524,628000.0,576000.0,52000.0,9.027778
9,030-019-672,567000.0,523000.0,44000.0,8.413002


## 6. Sample Validation Checks

Six validation checks confirm that the function produced correct outputs on the sample. No rows are removed or capped at this stage — outliers are documented, not excluded.

> **Why keep outliers?** Extreme value changes may reflect genuine events — redevelopment, subdivision, zoning reclassification — rather than data errors. Removing them before investigation could discard valid signal. They are flagged here for later review.

### Check 1 — Non-null and missing counts

In [7]:
print(f'{"Column":<40} {"Non-null":>10} {"Missing":>10} {"Missing %":>12}')
print('-' * 76)
for col in derived_cols:
    non_null = df[col].notna().sum()
    missing  = df[col].isna().sum()
    pct      = missing / len(df) * 100
    print(f'{col:<40} {non_null:>10,} {missing:>10,} {pct:>11.1f}%')

Column                                     Non-null    Missing    Missing %
----------------------------------------------------------------------------
current_total_assessed_value                    982         18         1.8%
previous_total_assessed_value                   972         28         2.8%
absolute_value_change                           972         28         2.8%
percentage_value_change                         972         28         2.8%


### Check 2 — Negative value counts (for totals)

In [8]:
for col in [COL_CURR_TOTAL, COL_PREV_TOTAL]:
    neg_count = (df[col] < 0).sum()
    status    = 'OK' if neg_count == 0 else f'WARNING: {neg_count} negative values'
    print(f'{col}: {status}')

current_total_assessed_value: OK
previous_total_assessed_value: OK


### Check 3 — Distribution statistics for absolute and percentage changes

In [9]:
print('absolute_value_change (CAD)')
print(f'  Min    : {df[COL_ABS_CHANGE].min():>18,.0f}')
print(f'  Median : {df[COL_ABS_CHANGE].median():>18,.0f}')
print(f'  Mean   : {df[COL_ABS_CHANGE].mean():>18,.0f}')
print(f'  Max    : {df[COL_ABS_CHANGE].max():>18,.0f}')
print()
print('percentage_value_change (%)')
print(f'  Min    : {df[COL_PCT_CHANGE].min():>18.4f}')
print(f'  Median : {df[COL_PCT_CHANGE].median():>18.4f}')
print(f'  Mean   : {df[COL_PCT_CHANGE].mean():>18.4f}')
print(f'  Max    : {df[COL_PCT_CHANGE].max():>18.4f}')

absolute_value_change (CAD)
  Min    :        -23,007,000
  Median :             23,000
  Mean   :            -15,321
  Max    :         28,016,000

percentage_value_change (%)
  Min    :           -61.6857
  Median :             2.0309
  Mean   :             1.9283
  Max    :            48.6034


### Check 4 — Infinity values

`inf` and `-inf` are not caught by `.isna()` and must be checked separately. Their presence would indicate that a zero denominator slipped through the eligibility mask.

In [10]:
for col in [COL_ABS_CHANGE, COL_PCT_CHANGE]:
    pos_inf = (df[col] == float('inf')).sum()
    neg_inf = (df[col] == float('-inf')).sum()
    total   = pos_inf + neg_inf
    status  = 'OK — no infinities' if total == 0 else f'WARNING: {total} infinity values'
    print(f'{col}: {status}')

absolute_value_change: OK — no infinities
percentage_value_change: OK — no infinities


### Check 5 — Extreme percentage changes (flagged for review, not removed)

Changes above +100% or below -50% are flagged. Common non-error causes: redevelopment, subdivision, land-use reclassification, or a very small prior-year denominator. These rows are documented here but not excluded from the dataset.

In [11]:
EXTREME_HIGH_THRESHOLD = 100.0   # percent
EXTREME_LOW_THRESHOLD  = -50.0   # percent

extreme_high = (df[COL_PCT_CHANGE] > EXTREME_HIGH_THRESHOLD).sum()
extreme_low  = (df[COL_PCT_CHANGE] < EXTREME_LOW_THRESHOLD).sum()

print(f'Changes above +{EXTREME_HIGH_THRESHOLD:.0f}% : {extreme_high:,}')
print(f'Changes below {EXTREME_LOW_THRESHOLD:.0f}%  : {extreme_low:,}')
print()

review_cols = [COL_PID, COL_CURR_TOTAL, COL_PREV_TOTAL, COL_ABS_CHANGE, COL_PCT_CHANGE]

print('Five largest percentage changes (for review):')
display(df.loc[df[COL_PCT_CHANGE].nlargest(5).index, review_cols])

print()
print('Five smallest percentage changes (for review):')
display(df.loc[df[COL_PCT_CHANGE].nsmallest(5).index, review_cols])

Changes above +100% : 0
Changes below -50%  : 2

Five largest percentage changes (for review):


,PID,current_total_assessed_value,previous_total_assessed_value,absolute_value_change,percentage_value_change
866,031-558-160,85658000.0,57642000.0,28016000.0,48.603449
233,031-595-669,737000.0,538067.0,198933.0,36.971790
230,013-522-868,3964000.0,3089326.0,874674.0,28.312778
555,010-659-111,5613000.0,4513000.0,1100000.0,24.374031
602,031-572-693,1717000.0,1403000.0,314000.0,22.380613



Five smallest percentage changes (for review):


,PID,current_total_assessed_value,previous_total_assessed_value,absolute_value_change,percentage_value_change
520,011-087-251,5223000.0,13632000.0,-8409000.0,-61.685739
518,009-273-999,13735800.0,28396600.0,-14660800.0,-51.628716
884,002-425-670,1518000.0,2930600.0,-1412600.0,-48.201733
530,015-527-301,2460000.0,4334000.0,-1874000.0,-43.239502
516,004-050-321,3360000.0,5553700.0,-2193700.0,-39.499793


### Check 6 — Arithmetic spot-check

For every row where both source totals are non-null, we independently re-compute the absolute change and percentage change using the raw formula and compare them to the derived columns. All values must match exactly.

In [12]:
both_valid   = df[COL_CURR_TOTAL].notna() & df[COL_PREV_TOTAL].notna()
eligible_pct = both_valid & (df[COL_PREV_TOTAL] > 0)

# Absolute change
expected_abs   = df.loc[both_valid, COL_CURR_TOTAL] - df.loc[both_valid, COL_PREV_TOTAL]
abs_match      = (expected_abs == df.loc[both_valid, COL_ABS_CHANGE]).all()

# Percentage change
expected_pct   = (
    df.loc[eligible_pct, COL_ABS_CHANGE] / df.loc[eligible_pct, COL_PREV_TOTAL] * 100
)
pct_match      = (expected_pct == df.loc[eligible_pct, COL_PCT_CHANGE]).all()

print(f'Absolute change arithmetic check — rows checked: {both_valid.sum():,}, all match: {abs_match}')
print(f'Percentage change arithmetic check — rows checked: {eligible_pct.sum():,}, all match: {pct_match}')

Absolute change arithmetic check — rows checked: 972, all match: True
Percentage change arithmetic check — rows checked: 972, all match: True


## 7. Planned Processed Outputs

The cells below define the intended export paths and the logic that will produce each output. **No export is performed in this notebook.** Full-dataset processing and export will be added once the sample logic above is confirmed correct and the output schemas are approved.

---

### Planned output 1: `property_value_change_summary.csv`

**Target path:** `data/processed/property_value_change_summary.csv`  
**Expected rows:** One row per year (or one row total, depending on the aggregation confirmed later)  
**Description:** Aggregate statistics (count, median, mean, min, max) for `absolute_value_change` and `percentage_value_change` across the full dataset, grouped by assessment year if a year column is available.

```python
# PLANNED — do not execute yet
# summary = (
#     df_full
#     .groupby('TAX_LEVY_YEAR')[[COL_ABS_CHANGE, COL_PCT_CHANGE]]
#     .agg(['count', 'median', 'mean', 'min', 'max'])
#     .reset_index()
# )
# summary.to_csv(PROCESSED_DATA_DIR / 'property_value_change_summary.csv', index=False)
```

---

### Planned output 2: `property_value_change_distribution.csv`

**Target path:** `data/processed/property_value_change_distribution.csv`  
**Expected rows:** One row per percentile bucket or histogram bin  
**Description:** Distribution of `percentage_value_change` across the full dataset, capturing the spread of assessed value movements across Vancouver properties. Useful for visualising the histogram of changes.

```python
# PLANNED — do not execute yet
# bins = pd.cut(df_full[COL_PCT_CHANGE], bins=20)
# distribution = bins.value_counts().sort_index().reset_index()
# distribution.columns = ['pct_change_bin', 'count']
# distribution.to_csv(PROCESSED_DATA_DIR / 'property_value_change_distribution.csv', index=False)
```

## 8. Final Status

This cell prints a clear confirmation of what this notebook currently does and does not do.

In [13]:
print('=' * 70)
print('NOTEBOOK STATUS')
print('=' * 70)
print()
print('This notebook currently validates sample logic ONLY.')
print(f'  Rows processed : {SAMPLE_ROWS:,} (sample — not the full raw dataset)')
print(f'  Raw file       : {RAW_DATA_PATH.name}')
print()
print('The full raw dataset has NOT been processed.')
print('No files have been written to data/processed/.')
print()
print('Next step: confirm sample output schemas, then apply')
print('compute_value_change_features() to the full dataset and export.')
print('=' * 70)

NOTEBOOK STATUS

This notebook currently validates sample logic ONLY.
  Rows processed : 1,000 (sample — not the full raw dataset)
  Raw file       : property_tax_report_raw.csv

The full raw dataset has NOT been processed.
No files have been written to data/processed/.

Next step: confirm sample output schemas, then apply
compute_value_change_features() to the full dataset and export.
